# PyTorch: costruire reti in pratica

Il codice del capitolo [«PyTorch: costruire reti in pratica»](https://book.paithon.it/main/PyTorch/overview.html), *Paithon Book*.

Le celle sono quelle del libro, nell'ordine in cui compaiono: il testo che le spiega sta nelle pagine, qui c'è solo la parte da eseguire e da rompere.

Generato da `scripts/genera-notebook.py`: le correzioni vanno fatte nelle pagine del libro, non qui.


> **Verificato il 2026-07-25** con torch 2.13.0, numpy 2.4.6, pandas 3.0.5, scikit-learn 1.9.0, transformers 5.14.1, diffusers 0.39.0, librosa 0.11.0, torch-geometric 2.8.0.post1. Tutte le celle di questo notebook sono state eseguite senza errori con quelle versioni; le librerie si muovono, e se qualcosa qui non gira piu' e' un errore del libro: [segnalalo](https://github.com/paithon-it/paithonbook/issues).


In [ ]:
# Su Colab quasi tutto c'è già; questa riga serve altrove.
%pip install -q numpy pillow torch torchinfo torchvision

In [ ]:
# Mostra il valore di ogni riga, come i commenti «# ->» del libro.
try:
    from IPython.core.interactiveshell import InteractiveShell
    InteractiveShell.ast_node_interactivity = 'all'
except ImportError:      # fuori da IPython non serve e non c'è
    pass

> **Cella di preparazione.** Crea i dati e i nomi che il testo da per esistenti. Non fa parte del libro: serve a far girare il notebook, e viene ripetuta all'inizio di ogni pagina perche ognuna riparta dallo stesso stato.


In [ ]:
_PRELUDIO = r'''
import pathlib

import torch
from PIL import Image

# --- un dataset di immagini in miniatura -------------------------------------
# La pagina "Dati su misura" parte da una cartella di fotografie proprie, con
# una sottocartella per classe. Qui quelle cartelle si creano: sei immagini
# minuscole generate a colori pieni, quel tanto che basta perche' ImageFolder,
# le trasformazioni e il DataLoader abbiano qualcosa da masticare.
COLORI = {"pizza": (200, 80, 40), "bistecca": (120, 40, 40), "sushi": (230, 220, 200)}
for parte in ("addestramento", "test"):
    for classe, colore in COLORI.items():
        cartella = pathlib.Path("dati") / parte / classe
        cartella.mkdir(parents=True, exist_ok=True)
        for n in range(3):
            Image.new("RGB", (64, 64), colore).save(cartella / f"img_{n}.jpg")

# --- i nomi che le pagine del capitolo danno per esistenti -------------------
# `dati` e `dati_test` compaiono nel testo senza essere costruiti: nel libro il
# punto e' l'API del DataLoader, non da dove arrivano i file.
from torchvision import datasets, transforms  # noqa: E402  (dopo la creazione dei file)

_preparazione = transforms.Compose([transforms.Resize((32, 32)), transforms.ToTensor()])
dati_train = datasets.ImageFolder(root="dati/addestramento", transform=_preparazione)
dati_test = datasets.ImageFolder(root="dati/test", transform=_preparazione)
dati = dati_train

# Un modello e un ingresso minimi, per i blocchi che li usano di passaggio.
modello = torch.nn.Sequential(torch.nn.Flatten(), torch.nn.Linear(3 * 32 * 32, 3))
x = torch.randn(2, 3, 32, 32)
'''
exec(_PRELUDIO)

## PyTorch: costruire reti in pratica

[Leggi la pagina](https://book.paithon.it/main/PyTorch/overview.html)


### Installazione e primo contatto


In [ ]:
import torch

print(torch.__version__)          # versione installata
print(torch.cuda.is_available())  # True se c'è una GPU NVIDIA utilizzabile

x = torch.tensor(3.0, requires_grad=True)  # un tensore "osservato"
y = x**2 + 2*x                             # y = x² + 2x, calcolato subito
y.backward()                               # gradiente automatico
print(x.grad)                              # la derivata di y in x=3 -> tensor(8.)

## Tensori e autograd

[Leggi la pagina](https://book.paithon.it/main/PyTorch/tensori.html)


In [ ]:
exec(_PRELUDIO)   # ripristina i nomi di partenza della pagina

### Creare tensori e farci i conti


In [ ]:
import torch

s = torch.tensor(3.14)                     # scalare, rank 0
v = torch.tensor([1.0, 2.0, 3.0])          # vettore, rank 1
M = torch.tensor([[1., 2.], [3., 4.]])     # matrice, rank 2

M.shape        # torch.Size([2, 2])
M.dtype        # torch.float32

torch.zeros(2, 3)        # matrice 2x3 di zeri
torch.ones(5)            # vettore di uno
torch.randn(3, 3)        # numeri a caso, quasi tutti fra -2 e 2, centrati sullo zero
torch.arange(0, 10, 2)   # da 0 a 10 di 2 in 2, 10 escluso: tensor([0, 2, 4, 6, 8])

In [ ]:
a = torch.tensor([1., 2., 3.])
b = torch.tensor([10., 20., 30.])

a + b            # tensor([11., 22., 33.])
a * b            # prodotto elemento per elemento
a.sum()          # tensor(6.)  -> calcolato SUBITO
a @ b            # prodotto scalare: 1·10 + 2·20 + 3·30 = tensor(140.)
a.reshape(3, 1)  # nuova forma: gli stessi numeri in colonna, 3x1

### Lo stesso codice su CPU e GPU


In [ ]:
# se una scheda grafica c'è usa quella, altrimenti la CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

M = torch.randn(1000, 1000)   # una matrice grande: un milione di numeri
M = M.to(device)              # trasloca dove dice `device`
prodotto = M @ M              # calcolato dove vive il tensore
print(prodotto.shape, prodotto.device)
# su una macchina senza scheda grafica: torch.Size([1000, 1000]) cpu

### Autograd: la derivata calcolata da sola


In [ ]:
x = torch.tensor(3.0, requires_grad=True)   # "osserva questo tensore"

y = x**2 + 2*x            # y = x² + 2x: il grafo si costruisce da solo
y.backward()              # passata all'indietro

x.grad                    # la derivata di y in x=3  ->  tensor(8.)

## Moduli: costruire il modello

[Leggi la pagina](https://book.paithon.it/main/PyTorch/moduli.html)


In [ ]:
exec(_PRELUDIO)   # ripristina i nomi di partenza della pagina

### `nn.Module`: il mattone di ogni rete


In [ ]:
import torch
from torch import nn

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()          # da griglia 28x28 a vettore 784
        self.hidden = nn.Linear(28 * 28, 128)  # ogni ingresso collegato a ogni neurone
        self.out = nn.Linear(128, 10)        # 10 uscite: una per cifra 0-9

    def forward(self, x):
        x = self.flatten(x)
        x = torch.relu(self.hidden(x))       # ReLU: i numeri negativi diventano zero
        return self.out(x)                   # punteggi grezzi, non probabilita'

model = MLP()
print(model)          # elenca i pezzi che compongono il modello
# MLP(
#   (flatten): Flatten(start_dim=1, end_dim=-1)
#   (hidden): Linear(in_features=784, out_features=128, bias=True)
#   (out): Linear(in_features=128, out_features=10, bias=True)
# )

### La scorciatoia: `nn.Sequential`


In [ ]:
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)

### Quanti parametri ha questa rete?


In [ ]:
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(n_params)   # 101770

### Misurare l'errore: le funzioni di perdita


In [ ]:
loss_regressione = nn.MSELoss()            # per predire numeri continui
loss_classi = nn.CrossEntropyLoss()        # per scegliere tra classi

# esempio: 2 immagini finte date in pasto al modello ancora ignorante.
# (2, 1, 28, 28) = 2 immagini, 1 canale (MNIST e' in scala di grigi), 28x28 pixel
logits = model(torch.randn(2, 1, 28, 28))  # shape (2, 10): 10 punteggi per immagine
target = torch.tensor([3, 7])              # le cifre vere sono un 3 e un 7
errore = loss_classi(logits, target)       # un numero solo: la loss media
print(errore.item())                       # circa 2,3 (con due sole immagini balla)

## Il training loop: addestrare un modello

[Leggi la pagina](https://book.paithon.it/main/PyTorch/addestramento.html)


In [ ]:
exec(_PRELUDIO)   # ripristina i nomi di partenza della pagina

### Il rito: i cinque passi


*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

for X_batch, y_batch in dataloader:
    y_pred = model(X_batch)             # 1. forward: la previsione
    loss = criterion(y_pred, y_batch)   # 2. loss: quanto abbiamo sbagliato
    optimizer.zero_grad()               # 3. via i gradienti del giro prima
    loss.backward()                     # 4. backward: calcola i gradienti
    optimizer.step()                    # 5. aggiorna i pesi
```


### `Dataset` e `DataLoader`: la catena di rifornimento


In [ ]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# MNIST scaricato e trasformato in tensori con valori in [0, 1]
train_data = datasets.MNIST(root="data", train=True, download=True,
                            transform=transforms.ToTensor())
test_data = datasets.MNIST(root="data", train=False, download=True,
                           transform=transforms.ToTensor())

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=256)   # niente shuffle, pacchetti piu' grandi

### MNIST da cima a fondo


In [ ]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- dati ---
train_data = datasets.MNIST(root="data", train=True, download=True,
                            transform=transforms.ToTensor())
test_data = datasets.MNIST(root="data", train=False, download=True,
                           transform=transforms.ToTensor())
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=256)

# --- modello, loss, ottimizzatore ---
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)   # 1e-3 e' 0,001

# --- addestramento ---
for epoca in range(5):
    model.train()                        # modalità addestramento
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        y_pred = model(X)
        loss = criterion(y_pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # --- valutazione a fine epoca ---
    model.eval()                         # modalità valutazione
    corretti = 0
    with torch.no_grad():                # niente gradienti: solo lettura
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            y_pred = model(X)
            # per ogni immagine prendi il punteggio piu' alto -> la cifra scelta;
            # confrontala con quella vera; conta i sì
            corretti += (y_pred.argmax(dim=1) == y).sum().item()

    print(f"epoca {epoca + 1}: accuratezza sul test {corretti / len(test_data):.3f}")

### Salvare il lavoro: lo `state_dict`


In [ ]:
torch.save(model.state_dict(), "mnist_mlp.pt")     # salva i numeri

model2 = nn.Sequential(                            # stessa architettura...
    nn.Flatten(), nn.Linear(28 * 28, 128), nn.ReLU(), nn.Linear(128, 10)
)
model2.load_state_dict(torch.load("mnist_mlp.pt")) # ...numeri ricaricati
model2.eval()                                      # pronto per l'uso

In [ ]:
from torch import optim

optimizer = optim.Adam(model.parameters(), lr=1e-3)

# checkpoint per RIPRENDERE: i pesi da soli non bastano
torch.save({"epoca": 5,
            "modello": model.state_dict(),
            "ottimizzatore": optimizer.state_dict()}, "checkpoint.pt")

stato = torch.load("checkpoint.pt")
model.load_state_dict(stato["modello"])
optimizer.load_state_dict(stato["ottimizzatore"])   # la riga che si dimentica
print(f"ripresa dall'epoca {stato['epoca']}; lo stato dell'ottimizzatore "
      f"ha le chiavi {list(stato['ottimizzatore'])}")
# ripresa dall'epoca 5; lo stato dell'ottimizzatore ha le chiavi ['state', 'param_groups']

## Il flusso di lavoro: dal problema al modello

[Leggi la pagina](https://book.paithon.it/main/PyTorch/flusso-di-lavoro.html)


In [ ]:
exec(_PRELUDIO)   # ripristina i nomi di partenza della pagina

### Un problema di cui conosciamo già la risposta


In [ ]:
import torch
from torch import nn

torch.manual_seed(42)          # stessi numeri casuali a ogni esecuzione

# I parametri "veri": il modello dovrà ritrovarli da solo, senza mai vederli.
peso_vero, bias_vero = 0.7, 0.3

X = torch.arange(0, 1, 0.02).unsqueeze(dim=1)   # 50 punti, shape (50, 1)
y = peso_vero * X + bias_vero                   # shape (50, 1)

taglio = int(0.8 * len(X))                      # 80% per addestrare, 20% per il test
X_train, y_train = X[:taglio], y[:taglio]       # (40, 1)
X_test,  y_test  = X[taglio:], y[taglio:]       # (10, 1)

In [ ]:
class RegressioneLineare(nn.Module):
    def __init__(self):
        super().__init__()
        self.strato = nn.Linear(in_features=1, out_features=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.strato(x)

modello = RegressioneLineare()
print(modello.state_dict())   # peso e bias, per ora casuali
# OrderedDict({'strato.weight': tensor([[0.7645]]), 'strato.bias': tensor([0.8300])})

In [ ]:
criterio = nn.L1Loss()                                     # errore assoluto medio
# lr e' il learning rate, il "passo" della sezione precedente
ottimizzatore = torch.optim.SGD(modello.parameters(), lr=0.01)

for epoca in range(1000):
    modello.train()
    y_pred = modello(X_train)
    perdita = criterio(y_pred, y_train)
    ottimizzatore.zero_grad()
    perdita.backward()
    ottimizzatore.step()

    if epoca % 199 == 0:                                   # il "termometro"
        modello.eval()
        with torch.no_grad():
            perdita_test = criterio(modello(X_test), y_test)
        print(f"epoca {epoca:>4} | train {perdita.item():.4f} "
              f"| test {perdita_test.item():.4f}")

print(modello.state_dict())

### Predire su dati nuovi: tre condizioni e due interruttori


In [ ]:
modello.eval()                                  # interruttore 1: modalità esame
with torch.no_grad():                           # interruttore 2: niente gradienti
    x_nuovo = torch.tensor([[0.95]],            # forma: (1, 1), non (1,)
                           dtype=torch.float32) # tipo: come in addestramento
    x_nuovo = x_nuovo.to(next(modello.parameters()).device)  # stesso dispositivo
    stima = modello(x_nuovo)
print(stima.item())        # ~ 0.7 * 0.95 + 0.3 = 0.965

## Dati su misura: `Dataset`, `DataLoader` e trasformazioni

[Leggi la pagina](https://book.paithon.it/main/PyTorch/dati-su-misura.html)


In [ ]:
exec(_PRELUDIO)   # ripristina i nomi di partenza della pagina

### La convenzione delle cartelle


In [ ]:
from torchvision import datasets, transforms

preparazione = transforms.Compose([
    transforms.Resize((224, 224)),   # tutte le immagini della stessa misura
    transforms.ToTensor(),           # da immagine a tensore (canali, altezza,
                                     # larghezza) con i valori portati fra 0 e 1
])

dati_train = datasets.ImageFolder(root="dati/addestramento", transform=preparazione)
dati_test = datasets.ImageFolder(root="dati/test", transform=preparazione)

print(dati_train.classes)         # ['bistecca', 'pizza', 'sushi']  (ordine alfabetico)
print(dati_train.class_to_idx)    # {'bistecca': 0, 'pizza': 1, 'sushi': 2}
print(len(dati_train))            # quante immagini in tutto
immagine, etichetta = dati_train[0]
print(immagine.shape, etichetta)  # torch.Size([3, 224, 224]) 0

### Scrivere un `Dataset` a mano


In [ ]:
import pathlib
import torch
from torch.utils.data import Dataset
from PIL import Image

class DatasetImmagini(Dataset):
    """Legge le immagini da cartelle-classe, come ImageFolder, ma è nostro."""

    def __init__(self, radice: str, transform=None):
        self.percorsi = sorted(pathlib.Path(radice).glob("*/*.jpg"))
        self.classi = sorted({p.parent.name for p in self.percorsi})
        self.classe_a_indice = {c: i for i, c in enumerate(self.classi)}
        self.transform = transform

    def __len__(self) -> int:
        return len(self.percorsi)

    def __getitem__(self, indice: int):
        percorso = self.percorsi[indice]
        immagine = Image.open(percorso).convert("RGB")     # anche i PNG a 4 canali
        etichetta = self.classe_a_indice[percorso.parent.name]
        if self.transform is not None:
            immagine = self.transform(immagine)
        return immagine, etichetta

### Le trasformazioni: preparare, e moltiplicare


In [ ]:
from torchvision import transforms

# ADDESTRAMENTO: prepara e moltiplica
train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),                    # ritaglio casuale
    transforms.RandomHorizontalFlip(p=0.5),        # specchiatura casuale
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],   # statistiche di ImageNet
                         std=[0.229, 0.224, 0.225]),
])

# VALUTAZIONE: solo prepara. Nessuna casualità.
test_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),                    # ritaglio deterministico
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

### Il `DataLoader` sul serio


In [ ]:
import os
from torch.utils.data import DataLoader

train_loader = DataLoader(
    dati_train,
    batch_size=32,
    shuffle=True,             # rimescola a ogni epoca: solo in addestramento
    num_workers=os.cpu_count(),   # processi che preparano i batch in parallelo
    pin_memory=True,          # memoria "bloccata": trasferimento più rapido alla GPU
    drop_last=True,           # scarta l'ultimo batch se incompleto
    persistent_workers=True,  # non li ricrea a ogni epoca
)

test_loader = DataLoader(dati_test, batch_size=64, shuffle=False,
                         num_workers=os.cpu_count(), pin_memory=True)

### Quando gli esempi non hanno la stessa forma: `collate_fn`


In [ ]:
import torch
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence

class DatasetSequenze(Dataset):
    """Frasi gia' tradotte in numeri, di lunghezza diversa fra loro."""

    def __init__(self, n=100):
        lunghezze = torch.randint(5, 40, (n,))
        self.esempi = [(torch.randint(1, 50, (int(l),)), int(l) % 2)
                       for l in lunghezze]

    def __len__(self):
        return len(self.esempi)

    def __getitem__(self, indice):
        return self.esempi[indice]

def raggruppa(batch):
    """Riceve una lista di (sequenza, etichetta); restituisce un batch imbottito."""
    sequenze, etichette = zip(*batch)
    lunghezze = torch.tensor([len(s) for s in sequenze])          # (B,)
    imbottite = pad_sequence(sequenze, batch_first=True,          # (B, L_max)
                             padding_value=0)
    return imbottite, lunghezze, torch.tensor(etichette)

dati = DatasetSequenze()
loader = DataLoader(dati, batch_size=32, shuffle=True, collate_fn=raggruppa)

imbottite, lunghezze, etichette = next(iter(loader))
print(imbottite.shape, lunghezze.shape, etichette.shape)
# le lunghezze vere sono tutte diverse, la larghezza del batch e' la massima:
print(lunghezze[:8].tolist(), "-> larghezza", imbottite.shape[1])

### Classi sbilanciate: pescare con criterio


In [ ]:
from torch.utils.data import WeightedRandomSampler

# Le etichette si leggono dall'indice, senza aprire una sola immagine:
# ImageFolder le tiene in .targets. Iterare il dataset le otterrebbe
# ugualmente, ma caricando tutti i file da disco, inutilmente.
etichette = torch.tensor(dati_train.targets)               # (N,)
conteggi = torch.bincount(etichette)                       # esempi per classe
peso_per_classe = 1.0 / conteggi.float()                   # la classe rara pesa di più
# indicizzare con un elenco: per ogni etichetta va a prendere il peso della sua
# classe, quindi da 3 pesi (uno per classe) si ottengono N pesi (uno per esempio)
pesi = peso_per_classe[etichette]                          # un peso per esempio

campionatore = WeightedRandomSampler(weights=pesi,
                                     num_samples=len(pesi),
                                     replacement=True)

# Attenzione: con un sampler NON si passa shuffle.
loader = DataLoader(dati_train, batch_size=32, sampler=campionatore)

### Dividere i dati senza barare


In [ ]:
import torch
from torch.utils.data import random_split

n_val = int(0.1 * len(dati_train))
n_train = len(dati_train) - n_val
generatore = torch.Generator().manual_seed(42)    # divisione riproducibile
sotto_train, sotto_val = random_split(dati_train, [n_train, n_val],
                                      generator=generatore)

## I tre errori più comuni

[Leggi la pagina](https://book.paithon.it/main/PyTorch/errori-comuni.html)


In [ ]:
exec(_PRELUDIO)   # ripristina i nomi di partenza della pagina

In [ ]:
x = torch.randn(32, 3, 224, 224)
print(x.shape)    # torch.Size([32, 3, 224, 224])  -> la forma
print(x.dtype)    # torch.float32                   -> il tipo
print(x.device)   # cpu                             -> dove abita

### 3. Il dispositivo non torna


*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

device = "cuda" if torch.cuda.is_available() else "cpu"
modello = modello.to(device)               # il modello trasloca...

for X, y in loader:
    X, y = X.to(device), y.to(device)      # ...e i dati devono seguirlo
    ...
```


In [ ]:
def forward(self, x):
    maschera = torch.ones(x.shape[-1])            # nasce su CPU: errore
    maschera = torch.ones(x.shape[-1], device=x.device)   # corretto

### Quando la validazione va meglio dell'addestramento


*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

# Registrare la storia costa due liste, e senza storia non c'e' diagnosi.
storia = {"train": [], "val": []}

for epoca in range(n_epoche):
    modello.train()
    somma, n = 0.0, 0
    for X, y in loader_train:
        ...                                   # i cinque passi soliti
        somma += perdita.item() * X.size(0)   # .item(): niente grafo in memoria
        n += X.size(0)
    storia["train"].append(somma / n)

    modello.eval()                            # niente dropout, niente augmentation
    with torch.no_grad():
        somma, n = 0.0, 0
        for X, y in loader_val:
            somma += loss_fn(modello(X), y).item() * X.size(0)
            n += X.size(0)
    storia["val"].append(somma / n)

    # stampare a ogni epoca, non alla fine: e' tutto il punto
    print(f"epoca {epoca:3d}  train {storia['train'][-1]:.4f}  val {storia['val'][-1]:.4f}")

import matplotlib.pyplot as plt
plt.plot(storia["train"], label="addestramento")
plt.plot(storia["val"], label="validazione", linestyle="--")
plt.xlabel("epoche"); plt.ylabel("loss"); plt.legend()
```


## Dal notebook agli script

[Leggi la pagina](https://book.paithon.it/main/PyTorch/dal-notebook-agli-script.html)


In [ ]:
exec(_PRELUDIO)   # ripristina i nomi di partenza della pagina

### Cinque file, cinque responsabilità


In [ ]:
# engine.py
import torch

def passo_addestramento(modello, loader, criterio, ottimizzatore, device):
    """Una epoca di addestramento. Restituisce (perdita media, accuratezza)."""
    modello.train()
    perdita_tot, corretti, totale = 0.0, 0, 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logit = modello(X)
        perdita = criterio(logit, y)

        ottimizzatore.zero_grad()
        perdita.backward()
        ottimizzatore.step()

        perdita_tot += perdita.item() * X.size(0)   # .item(): niente grafo trattenuto
        corretti += (logit.argmax(dim=1) == y).sum().item()
        totale += X.size(0)

    return perdita_tot / totale, corretti / totale


@torch.no_grad()                                    # decoratore: niente gradienti qui dentro
def passo_valutazione(modello, loader, criterio, device):
    """Una epoca di valutazione. Stessa firma, nessun aggiornamento dei pesi."""
    modello.eval()
    perdita_tot, corretti, totale = 0.0, 0, 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logit = modello(X)
        perdita_tot += criterio(logit, y).item() * X.size(0)
        corretti += (logit.argmax(dim=1) == y).sum().item()
        totale += X.size(0)

    return perdita_tot / totale, corretti / totale

### Il punto d'ingresso


*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

# train.py
import argparse
import torch
from torch import nn

import data_setup, engine, model_builder, utils

def main() -> None:
    p = argparse.ArgumentParser(description="Addestra un classificatore di immagini.")
    p.add_argument("--dati", type=str, required=True, help="cartella con train/ e test/")
    p.add_argument("--epoche", type=int, default=10)
    p.add_argument("--batch", type=int, default=32)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--unita-nascoste", type=int, default=128)
    p.add_argument("--seme", type=int, default=42)
    p.add_argument("--uscita", type=str, default="modelli/modello.pt")
    args = p.parse_args()

    utils.fissa_seme(args.seme)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    train_loader, test_loader, classi = data_setup.crea_dataloader(
        radice=args.dati, batch_size=args.batch)

    modello = model_builder.CNNSemplice(
        unita_nascoste=args.unita_nascoste, n_classi=len(classi)).to(device)

    criterio = nn.CrossEntropyLoss()
    ottimizzatore = torch.optim.Adam(modello.parameters(), lr=args.lr)

    for epoca in range(args.epoche):
        pt, at = engine.passo_addestramento(modello, train_loader, criterio,
                                            ottimizzatore, device)
        pv, av = engine.passo_valutazione(modello, test_loader, criterio, device)
        print(f"epoca {epoca+1:>2}/{args.epoche} | "
              f"train perdita {pt:.4f} acc {at:.3f} | "
              f"test perdita {pv:.4f} acc {av:.3f}")

    utils.salva_modello(modello, ottimizzatore, args.epoche, args.uscita,
                        classi=classi, argomenti=vars(args))

if __name__ == "__main__":      # eseguito solo se si lancia questo file
    main()
```


In [ ]:
# utils.py
import pathlib, torch

def salva_modello(modello, ottimizzatore, epoca, percorso, classi, argomenti):
    """Un checkpoint completo: per *usare* il modello e per *riprendere* il lavoro."""
    percorso = pathlib.Path(percorso)
    percorso.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"pesi": modello.state_dict(),
                "ottimizzatore": ottimizzatore.state_dict(),
                "epoca": epoca,
                "classi": classi,
                "config": argomenti}, percorso)

### Riproducibilità: fissare il caso


In [ ]:
# utils.py
import random
import numpy as np
import torch

def fissa_seme(seme: int = 42) -> None:
    random.seed(seme)             # librerie standard
    np.random.seed(seme)          # NumPy (usato dalle trasformazioni)
    torch.manual_seed(seme)       # PyTorch, CPU
    torch.cuda.manual_seed_all(seme)   # PyTorch, tutte le GPU

In [ ]:
torch.use_deterministic_algorithms(True)   # errore se un'op non ha versione deterministica
torch.backends.cudnn.benchmark = False     # niente autotuning degli algoritmi
# e, per cuBLAS, la variabile d'ambiente CUBLAS_WORKSPACE_CONFIG=:4096:8

## Replicare un paper

[Leggi la pagina](https://book.paithon.it/main/PyTorch/replicare-un-paper.html)


In [ ]:
exec(_PRELUDIO)   # ripristina i nomi di partenza della pagina

### Il caso: il Vision Transformer


In [ ]:
import torch
from torch import nn

class IncorporazionePatch(nn.Module):
    """Equazione 1 del paper: da immagine a sequenza di token."""

    def __init__(self, canali=3, patch=16, d_modello=768, immagine=224):
        super().__init__()
        n_patch = (immagine // patch) ** 2                    # 196

        # Il trucco: una convoluzione con kernel = stride = patch È la
        # proiezione lineare delle patch appiattite, calcolata tutta insieme.
        self.proiezione = nn.Conv2d(canali, d_modello,
                                    kernel_size=patch, stride=patch)

        self.token_classe = nn.Parameter(torch.randn(1, 1, d_modello))
        self.posizioni = nn.Parameter(torch.randn(1, n_patch + 1, d_modello))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B = x.shape[0]
        x = self.proiezione(x)                    # (B, 3, 224, 224) -> (B, 768, 14, 14)
        x = x.flatten(2).transpose(1, 2)          #                  -> (B, 196, 768)
        cls = self.token_classe.expand(B, -1, -1) #                     (B, 1, 768)
        x = torch.cat([cls, x], dim=1)            #                  -> (B, 197, 768)
        return x + self.posizioni                 # broadcast su tutto il batch

In [ ]:
class BloccoTransformer(nn.Module):
    """Equazioni 2 e 3: attenzione multi-testa e MLP, entrambe pre-norm."""

    def __init__(self, d_modello=768, teste=12, d_mlp=3072, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_modello)
        self.attenzione = nn.MultiheadAttention(d_modello, teste,
                                                dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_modello)
        self.mlp = nn.Sequential(
            nn.Linear(d_modello, d_mlp),
            nn.GELU(),                       # il paper usa GELU, non ReLU
            nn.Dropout(dropout),
            nn.Linear(d_mlp, d_modello),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.norm1(x)
        x = x + self.attenzione(h, h, h, need_weights=False)[0]   # residuo 1
        x = x + self.mlp(self.norm2(x))                           # residuo 2
        return x

### Verificare senza addestrare


In [ ]:
modello = nn.Sequential(
    IncorporazionePatch(),
    *[BloccoTransformer() for _ in range(12)],
)

finto = torch.randn(2, 3, 224, 224)               # due immagini finte
uscita = modello(finto)
print(uscita.shape)                                # torch.Size([2, 197, 768])

n_parametri = sum(p.numel() for p in modello.parameters() if p.requires_grad)
print(f"{n_parametri:,}")                          # 85,797,120

In [ ]:
from torchinfo import summary
summary(modello, input_size=(1, 3, 224, 224),
        col_names=["input_size", "output_size", "num_params"])

## Prestazioni e scala: spremere l'hardware

[Leggi la pagina](https://book.paithon.it/main/PyTorch/prestazioni.html)


In [ ]:
exec(_PRELUDIO)   # ripristina i nomi di partenza della pagina

### Metà dei byte, quasi doppia velocità: la precisione mista


In [ ]:
import torch
from torch import nn

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
# su GPU si usa float16 con la "lente"; su CPU l'autocast lavora in bfloat16,
# che della lente non ha bisogno (vedi il testo dopo il blocco)
mezza = torch.float16 if dispositivo == "cuda" else torch.bfloat16

model = nn.Sequential(nn.Flatten(), nn.Linear(28 * 28, 128), nn.ReLU(),
                      nn.Linear(128, 10)).to(dispositivo)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
train_loader = [(torch.randn(16, 1, 28, 28), torch.randint(0, 10, (16,)))
                for _ in range(3)]               # tre batch finti, per far girare il ciclo

scaler = torch.amp.GradScaler(dispositivo)       # la "lente" per i gradienti

for i, (X, y) in enumerate(train_loader):
    X, y = X.to(dispositivo), y.to(dispositivo)
    optimizer.zero_grad()
    with torch.autocast(dispositivo, dtype=mezza):
        y_pred = model(X)                        # forward in mezza precisione
        loss = criterion(y_pred, y)
    scaler.scale(loss).backward()                # loss amplificata, poi backward
    scaler.step(optimizer)                       # gradienti riportati in scala
    scaler.update()                              # ricalibra il fattore di scala
    print(f"batch {i}: loss {loss.item():.4f} | tipo interno {y_pred.dtype}")

### Misurare davvero: la coda asincrona


In [ ]:
import time
import torch

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
A = torch.randn(1024, 1024, device=dispositivo)

def cronometra(sincronizza):
    """Il parametro decide se aspettare la GPU alla fine: True sì, False no."""
    if dispositivo == "cuda":
        torch.cuda.synchronize()          # parti da una coda vuota
    t0 = time.perf_counter()
    for _ in range(10):
        B = A @ A
    if sincronizza and dispositivo == "cuda":
        torch.cuda.synchronize()          # aspetta che la GPU abbia finito DAVVERO
    return time.perf_counter() - t0

for _ in range(3):                        # riscaldamento, fuori dal cronometro
    A @ A

print(f"dispositivo: {dispositivo}")
senza = min(cronometra(False) for _ in range(3))   # il minimo, non la prima misura
con   = min(cronometra(True)  for _ in range(3))
print(f"senza synchronize: {senza * 1000:8.2f} ms")
print(f"con synchronize  : {con   * 1000:8.2f} ms")
if dispositivo == "cpu":
    print(f"(su CPU non c'è coda asincrona: i due numeri sono dello stesso "
          f"ordine, qui a {abs(con - senza) / senza:.0%} di distanza)")

### Una riga per compilare: `torch.compile`


In [ ]:
model = torch.compile(model)   # tutto qui: il resto del codice non cambia

### Più GPU, un solo modello: il parallelismo dati


*Frammento illustrativo: nel libro mostra la forma, qui non si esegue.*

```python

# SCHEMA, si lancia con: torchrun --nproc_per_node=4 addestra.py
import os
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler

dist.init_process_group("nccl")                 # collega i 4 processi
rank = int(os.environ["LOCAL_RANK"])            # chi sono io? (0, 1, 2 o 3)
torch.cuda.set_device(rank)

model = DDP(model.to(rank), device_ids=[rank])  # replica sincronizzata

sampler = DistributedSampler(train_data)        # a ognuno la sua fetta
loader = DataLoader(train_data, batch_size=64, sampler=sampler)

for epoca in range(epoche):
    sampler.set_epoch(epoca)                    # rimescola in modo coordinato
    for X, y in loader:
        ...                                     # training loop IDENTICO:
                                                # l'all-reduce avviene da solo
                                                # dentro loss.backward()
dist.destroy_process_group()
```


### Partire col piede giusto: `nn.init`


In [ ]:
from torch import nn

def inizializza(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")  # ricetta He
        nn.init.zeros_(m.bias)

model.apply(inizializza)   # applica la funzione a ogni sotto-modulo